# Image-only baseline \u2014 Subtask A1

Fine-tune a Vision Transformer (ViT) on the meme images alone (text is ignored). Jupyter version of [`baselines/train_image.py`](../train_image.py).

In [1]:
import sys, json, os, random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification, Trainer, TrainingArguments

TASK1 = Path.cwd().resolve()
while TASK1.name != 'taskA' and TASK1.parent != TASK1:
    TASK1 = TASK1.parent
sys.path.insert(0, str(TASK1 / 'baselines'))

from io_utils import read_jsonl, write_subtask_a1_tsv
from labels import get_task

DATA = TASK1 / 'data'
SUBTASK = 'a1'
MODEL_NAME = 'google/vit-base-patch16-224'
EPOCHS = 1
BATCH = 32
LR = 3e-4
SEED = 42
RUN_ID = 'vit_image_notebook'
OUT = TASK1 / 'predictions' / 'image_a1.tsv'

for fn in (random.seed, np.random.seed, torch.manual_seed):
    fn(SEED)
torch.cuda.manual_seed_all(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

device: cpu


/export/home/alishahrour/anaconda3/envs/ArGuard/lib/python3.11/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12090). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


## 1. Load splits

In [2]:
spec = get_task(SUBTASK)
train_records = read_jsonl(DATA / 'splits' / 'train.jsonl')
dev_records = read_jsonl(DATA / 'splits' / 'dev.jsonl')
target_records = read_jsonl(DATA / 'splits' / 'dev_test.jsonl')
print(f'train={len(train_records)}  dev={len(dev_records)}  target={len(target_records)}')

train=3500  dev=500  target=500


## 2. Dataset

In [3]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

class ImgDS(Dataset):
    def __init__(self, records, spec, processor, data_dir):
        self.records = records; self.spec = spec; self.proc = processor; self.data_dir = data_dir
    def __len__(self): return len(self.records)
    def __getitem__(self, i):
        r = self.records[i]
        p = self.data_dir / r['image_path']
        try:
            img = Image.open(p).convert('RGB')
        except Exception:
            img = Image.new('RGB', (224, 224), color=(0, 0, 0))
        px = self.proc(img, return_tensors='pt')['pixel_values'].squeeze(0)
        label = r.get('label')
        return {'pixel_values': px, 'labels': -100 if label is None else self.spec.label2id[label]}

train_ds = ImgDS(train_records, spec, processor, DATA)
dev_ds = ImgDS(dev_records, spec, processor, DATA)
target_ds = ImgDS(target_records, spec, processor, DATA)
print('built datasets')

built datasets


## 3. Model and Trainer

In [4]:
model = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=spec.num_labels,
    id2label={int(i): n for i, n in spec.id2label.items()},
    label2id=dict(spec.label2id),
    problem_type=spec.problem_type,
    ignore_mismatched_sizes=True,
)

training_args = TrainingArguments(
    output_dir=str(TASK1 / 'results' / 'notebook_image'),
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=2*BATCH,
    eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
    num_train_epochs=EPOCHS, learning_rate=LR, warmup_ratio=0.0, weight_decay=0.0,
    logging_steps=50, save_total_limit=1, push_to_hub=False, report_to='none',
    metric_for_best_model='eval_loss', greater_is_better=False,
    seed=SEED, data_seed=SEED, remove_unused_columns=False, dataloader_num_workers=2,
)

def collate(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'labels': torch.tensor([b['labels'] for b in batch], dtype=torch.long),
    }

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=dev_ds,
    data_collator=collate, processing_class=processor,
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 4. Train and predict

In [5]:
trainer.train()
print('dev metrics:', trainer.evaluate())

/export/home/alishahrour/anaconda3/envs/ArGuard/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.309005,1.201574


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['vit.layers.0.attention.q_proj.weight', 'vit.layers.0.attention.q_proj.bias', 'vit.layers.0.attention.k_proj.weight', 'vit.layers.0.attention.k_proj.bias', 'vit.layers.0.attention.v_proj.weight', 'vit.layers.0.attention.v_proj.bias', 'vit.layers.0.attention.o_proj.weight', 'vit.layers.0.attention.o_proj.bias', 'vit.layers.0.layernorm_before.weight', 'vit.layers.0.layernorm_before.bias', 'vit.layers.0.layernorm_after.weight', 'vit.layers.0.layernorm_after.bias', 'vit.layers.0.mlp.fc1.weight', 'vit.layers.0.mlp.fc1.bias', 'vit.layers.0.mlp.fc2.weight', 'vit.layers.0.mlp.fc2.bias', 'vit.layers.1.attention.q_proj.weight', 'vit.layers.1.attention.q_proj.bias', 'vit.layers.1.attention.k_proj.weight', 'vit.layers.1.attention.k_proj.bias', 'vit.layers.1.attention.v_proj.weight', 'vit.layers.1.attention.v_proj.bias', 'vit.layers.1.attention.o_proj.weight', 'vit.layers.1.attention.o_proj.bias', 'vit.layers.1.layernorm_before

Training Loss,Validation Loss,Epoch
1.309005,1.201574,1


dev metrics: {'eval_loss': 1.2015736103057861}


In [6]:
p = trainer.predict(target_ds)
logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
pred_ids = logits.argmax(axis=1)
rows = [(r['id'], spec.id2label[int(pi)]) for r, pi in zip(target_records, pred_ids)]
OUT.parent.mkdir(parents=True, exist_ok=True)
write_subtask_a1_tsv(rows, OUT, run_id=RUN_ID)
print('wrote', OUT)

wrote /export/home/alishahrour/Shared_Task/ArGuard-2026-tasks/taskA/predictions/image_a1.tsv


## 5. Validate

In [7]:
import subprocess
print(subprocess.check_output(
    [sys.executable, str(TASK1 / 'format_checker' / 'format_checker.py'),
     '--subtask', SUBTASK, '--predictions', str(OUT)],
    text=True,
))

OK



INFO [subtask_a1] OK: 500 predictions, run_id='vit_image_notebook', labels in ['Hateful', 'Not Hateful']
